In [1]:
import numpy as np
import scipy.sparse as sp
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd
import os
import gc
import zipfile
from numba import njit
from tqdm import tqdm

# ==========================================
# 1. ネットワークと単体複体生成 (Numba JIT)
# ==========================================
@njit
def build_simplicial_complex(N, m, seed=42):
    np.random.seed(seed)
    num_edges = 2 * m * N
    row = np.empty(num_edges, dtype=np.int32)
    col = np.empty(num_edges, dtype=np.int32)
    targets = np.empty(num_edges, dtype=np.int32)
    max_faces = m * m * N
    faces = np.empty((max_faces, 3), dtype=np.int32)

    idx, target_idx, face_idx = 0, 0, 0

    for i in range(m):
        for j in range(m):
            if i != j:
                row[idx] = i; col[idx] = j; idx += 1
                targets[target_idx] = i; target_idx += 1

    if m >= 3:
        faces[face_idx, 0] = 0; faces[face_idx, 1] = 1; faces[face_idx, 2] = 2; face_idx += 1

    chosen = np.empty(m, dtype=np.int32)

    for i in range(m, N):
        c_idx = 0
        while c_idx < m:
            r = np.random.randint(0, target_idx)
            node = targets[r]
            already = False
            for k in range(c_idx):
                if chosen[k] == node:
                    already = True; break
            if not already:
                chosen[c_idx] = node; c_idx += 1

        for j in range(m):
            for k in range(j + 1, m):
                faces[face_idx, 0] = i
                faces[face_idx, 1] = chosen[j]
                faces[face_idx, 2] = chosen[k]
                face_idx += 1

        for k in range(m):
            target = chosen[k]
            row[idx] = i; col[idx] = target; idx += 1
            row[idx] = target; col[idx] = i; idx += 1
            targets[target_idx] = i; target_idx += 1
            targets[target_idx] = target; target_idx += 1

    return row[:idx], col[:idx], faces[:face_idx]

@njit
def evaluate_faces(X, faces, active_faces, theta_face):
    ruptured = 0
    for f in range(faces.shape[0]):
        if active_faces[f]:
            i, j, k = faces[f, 0], faces[f, 1], faces[f, 2]
            frustration = max(X[i], X[j], X[k]) - min(X[i], X[j], X[k])
            if frustration > theta_face:
                active_faces[f] = False
                ruptured += 1
    return ruptured

# ==========================================
# 2. 動的ネットワーク(適応的エッジ切断)シミュレーション
# ==========================================
def run_dynamic_network_simulation(N, m, params, seed=42):
    np.random.seed(seed)
    dt = params['dt']
    T1, T2, T3 = params['T1'], params['T2'], params['T3']
    theta_face = params['theta_face']
    theta_sever = params['theta_sever']
    p_sever = params['p_sever']

    row, col, faces = build_simplicial_complex(N, m, seed)
    data = np.ones(len(row), dtype=np.float32)

    # 初期次数からハブを特定
    A_sparse = sp.csr_matrix((data, (row, col)), shape=(N, N))
    degrees = np.array(A_sparse.sum(axis=1)).flatten()
    hub_node = np.argmax(degrees)

    X = np.random.uniform(0, params['epsilon'], N).astype(np.float32)
    b = np.zeros(N, dtype=np.float32)

    num_faces = faces.shape[0]
    num_edges = len(row)
    active_faces = np.ones(num_faces, dtype=bool)

    history_betti_1 = []      # 面の崩壊（β1相当）
    history_rewired = []      # 線の切断・再接続累積（β0動態相当）
    history_ks_stat = []

    total_ruptured_faces = 0
    cumulative_rewired_edges = 0
    X_prev = np.copy(X)

    for step in tqdm(range(T3), desc=f"Simulating N={N}", leave=False):
        if step == T1:
            b[hub_node] = 0.9
        elif step == T2:
            b[hub_node] = 0.0

        # 1. ネットワーク力学系による状態更新
        A_sparse = sp.csr_matrix((data, (row, col)), shape=(N, N))
        degrees = np.array(A_sparse.sum(axis=1)).flatten()
        degrees[degrees == 0] = 1.0

        P = A_sparse.dot(X) / degrees
        noise = np.random.normal(0, 1, N).astype(np.float32) * np.sqrt(dt)
        dX = (P + b - X) * dt + params['sigma'] * noise
        X = np.clip(X + dX, 0, 1)

        # 2. 面の崩壊評価（β1）
        total_ruptured_faces += evaluate_faces(X, faces, active_faces, theta_face)

        # 3. 動的エッジ評価（線の切断と再接続）
        # フラストレーションが極限（theta_sever）を超えたエッジを確率的に切断
        frustrations = np.abs(X[row] - X[col])
        sever_mask = frustrations > theta_sever

        if np.any(sever_mask):
            sever_candidates = np.where(sever_mask)[0]
            # p_severの確率で実際に切断
            actual_sever = sever_candidates[np.random.random(len(sever_candidates)) < p_sever]

            if len(actual_sever) > 0:
                cumulative_rewired_edges += len(actual_sever)
                # 切断されたエッジをランダムな他ノードに再接続（Rewiring）
                col[actual_sever] = np.random.randint(0, N, size=len(actual_sever))

        # 履歴記録
        history_betti_1.append(total_ruptured_faces / max(1, num_faces))
        history_rewired.append(cumulative_rewired_edges / max(1, num_edges))

        # Burn-in期間のKS検定
        if step <= T1:
            if step % 10 == 0:
                ks_stat, _ = stats.ks_2samp(X[::max(1, N//1000)], X_prev[::max(1, N//1000)])
                history_ks_stat.append((step, ks_stat))
                X_prev = np.copy(X)

    return history_betti_1, history_rewired, history_ks_stat

# ==========================================
# 3. メイン実行スクリプト
# ==========================================
def main():
    N_list = [1000, 10000, 100000, 1000000]

    params = {
        'dt': 0.1,
        'T1': 100,             # Burn-in 終了
        'T2': 250,             # Pathogenesis 終了 (少し長めにして切断を引き起こす)
        'T3': 400,             # Hysteresis 確認 終了
        'epsilon': 0.05,
        'sigma': 0.02,
        'theta_face': 0.25,    # 面が破裂する閾値 (早い)
        'theta_sever': 0.45,   # 線が切断される閾値 (遅い・高い)
        'p_sever': 0.05        # 切断確率
    }

    output_dir = "Advanced_Dynamic_Network"
    os.makedirs(output_dir, exist_ok=True)
    summary_data = []

    for N in N_list:
        try:
            print(f"\n[{N:,} Nodes] 動的ネットワーク・タイムラグ検証開始...")
            b1_hist, rewire_hist, ks_hist = run_dynamic_network_simulation(N, 3, params)

            # 1. KS検定による自己組織化証明
            steps_ks, ks_vals = zip(*ks_hist)
            plt.figure(figsize=(8, 5))
            plt.plot(steps_ks, ks_vals, 'g-o', linewidth=2)
            plt.title(f'Convergence to Empirical Stationary Distribution (N={N})')
            plt.xlabel('Burn-in Steps')
            plt.ylabel('KS Statistic (Distribution Drift)')
            plt.grid(True, alpha=0.3)
            plt.savefig(os.path.join(output_dir, f'Adv_Dynamic_KS_N{N}.png'), dpi=300)
            plt.close()

            # 2. 面の崩壊と線の切断のタイムラグ比較プロット
            df = pd.DataFrame({
                'TimeStep': np.arange(params['T3']),
                'Face_Rupture_Betti1': b1_hist,
                'Edge_Severing_Rewire': rewire_hist
            })
            df.to_csv(os.path.join(output_dir, f'Adv_dynamic_lag_N{N}.csv'), index=False)

            fig, ax1 = plt.subplots(figsize=(12, 6))

            ax1.set_xlabel('Time Step')
            ax1.set_ylabel('Simplicial Face Rupture (Normalized $\Delta \\beta_1$)', color='red')
            ax1.plot(df['TimeStep'], df['Face_Rupture_Betti1'], 'r-', linewidth=2.5, label='Community Collapse (Faces ruptured)')
            ax1.tick_params(axis='y', labelcolor='red')

            ax2 = ax1.twinx()
            ax2.set_ylabel('Edge Severing & Rewiring (Normalized Disconnections)', color='blue')
            ax2.plot(df['TimeStep'], df['Edge_Severing_Rewire'], 'b--', linewidth=2.5, label='Individual Unfollowing (Edges severed)')
            ax2.tick_params(axis='y', labelcolor='blue')

            # フェーズの背景
            ax1.axvspan(0, params['T1'], color='gray', alpha=0.1, label='Phase 0: Burn-in')
            ax1.axvspan(params['T1'], params['T2'], color='red', alpha=0.1, label='Phase 1a: Pathogenesis')
            ax1.axvspan(params['T2'], params['T3'], color='blue', alpha=0.1, label='Phase 1b: Hysteresis')

            fig.tight_layout()
            plt.title(f'Time-Lagged Rupture: Community Consensus vs Individual Ties (N={N:,})')

            # 凡例をまとめる
            lines_1, labels_1 = ax1.get_legend_handles_labels()
            lines_2, labels_2 = ax2.get_legend_handles_labels()
            ax1.legend(lines_1 + lines_2, labels_1 + labels_2, loc='upper left')

            plt.savefig(os.path.join(output_dir, f'Adv_TimeLag_Plot_N{N}.png'), dpi=300)
            plt.close()

            summary_data.append({
                'N': N,
                'Final_Betti1_Ratio': b1_hist[-1],
                'Final_Rewire_Ratio': rewire_hist[-1]
            })
            gc.collect()

        except MemoryError:
            print(f"[警告] N={N} でメモリ不足。")
            break

    # 3. まとめ出力とZIP化
    if summary_data:
        df_sum = pd.DataFrame(summary_data)
        df_sum.to_csv(os.path.join(output_dir, 'Adv_dynamic_summary.csv'), index=False)

        plt.figure(figsize=(8, 6))
        plt.plot(df_sum['N'], df_sum['Final_Betti1_Ratio'], 'r-o', linewidth=2.5, label='Total Face Ruptures ($\Delta \\beta_1$)')
        plt.plot(df_sum['N'], df_sum['Final_Rewire_Ratio'], 'b--s', linewidth=2.5, label='Total Edge Severings (Rewiring)')
        plt.xscale('log')
        plt.title('Asymptotic Damage Scale in Dynamic Networks')
        plt.xlabel('Network Size N (Log Scale)')
        plt.ylabel('Normalized Structural Damage Proportion')
        plt.legend()
        plt.grid(True, alpha=0.3)
        plt.savefig(os.path.join(output_dir, 'Adv_dynamic_summary_plot.png'), dpi=300)
        plt.close()

        zip_filename = 'Advanced_Validation_Dynamic_Network.zip'
        with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
            for root, _, files in os.walk(output_dir):
                for file in files:
                    zipf.write(os.path.join(root, file), arcname=file)
        print(f"\n🎉 動的ネットワーク検証の計算完了！結果は '{zip_filename}' に出力されました。")

if __name__ == "__main__":
    main()

<>:208: SyntaxWarning: invalid escape sequence '\D'
<>:250: SyntaxWarning: invalid escape sequence '\D'
<>:208: SyntaxWarning: invalid escape sequence '\D'
<>:250: SyntaxWarning: invalid escape sequence '\D'
/tmp/ipykernel_2072/4052541091.py:208: SyntaxWarning: invalid escape sequence '\D'
  ax1.set_ylabel('Simplicial Face Rupture (Normalized $\Delta \\beta_1$)', color='red')
/tmp/ipykernel_2072/4052541091.py:250: SyntaxWarning: invalid escape sequence '\D'
  plt.plot(df_sum['N'], df_sum['Final_Betti1_Ratio'], 'r-o', linewidth=2.5, label='Total Face Ruptures ($\Delta \\beta_1$)')



[1,000 Nodes] 動的ネットワーク・タイムラグ検証開始...



[10,000 Nodes] 動的ネットワーク・タイムラグ検証開始...



[100,000 Nodes] 動的ネットワーク・タイムラグ検証開始...



[1,000,000 Nodes] 動的ネットワーク・タイムラグ検証開始...



🎉 動的ネットワーク検証の計算完了！結果は 'Advanced_Validation_Dynamic_Network.zip' に出力されました。
